# Transformer pruning study — Kaggle / Colab runner

Runs `experiments/research/run_study.py` from the `structured-pruning` repo on a free GPU.
Full instructions: `docs/RUNNING_EXPERIMENTS.md` in the repo.

**Before running**

- **Kaggle:** Settings → Accelerator → **GPU T4 x2** or **GPU P100**. Settings → **Internet on** (this needs a phone-verified account). For long runs, use **Save Version → Save & Run All (Commit)**, which keeps running after you close the tab.
- **Colab:** Runtime → Change runtime type → **T4 GPU**. Results are written to your Google Drive, so a disconnect loses at most the run in progress.

Edit the **Configuration** cell, then run all cells. Re-running continues where the last session stopped.

## 1. Configuration

In [ ]:
# ---- what to run -----------------------------------------------------------
TASKS      = ["mrpc"]                         # mrpc rte sst2 qnli cola stsb qqp mnli
MODEL      = "google-bert/bert-base-uncased"  # or "prajjwal1/bert-small" for a cheap dry run --
                                             # on transformers 5 that also needs
                                             # EXTRA_ARGS = ["--tokenizer", "google-bert/bert-base-uncased"]
SEEDS      = [1, 2, 3]
VARIANTS   = ["quick"]                        # presets: quick, rq1-ffn, rq1-heads, rq2-order, all
PERCENTS   = [20, 40, 60]
EPOCHS     = 3                                # baseline fine-tuning
HEAL_EPOCHS   = 1                             # fine-tuning after each cut (0 = pre-heal only)
HEAL_EXAMPLES = None                          # e.g. 8000 to cap healing cost on sst2/qnli
EXTRA_ARGS = []                               # e.g. ["--latency"] or ["--lr", "3e-5"]

# ---- where the code comes from ---------------------------------------------
REPO_URL = "https://github.com/thakerpragnesh/structured-pruning.git"
BRANCH   = "main"

# ---- session limits --------------------------------------------------------
# Stop starting new runs after this long, so the session ends cleanly before
# the platform kills it (Kaggle: ~12 h; Colab free: often less).
TIME_BUDGET_HOURS = 11.0

## 2. Platform, working directory, resume source

In [ ]:
import glob, os, sys
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
ON_COLAB = "google.colab" in sys.modules

if ON_KAGGLE:
    WORK_DIR = Path("/kaggle/working/study_runs")
    CODE_DIR = Path("/kaggle/working/structured-pruning")
    # A previous session's output, attached via "Add Input" -> your notebook's output.
    found = sorted(glob.glob("/kaggle/input/*/study_runs/results.jsonl")) + \
            sorted(glob.glob("/kaggle/input/*/*/study_runs/results.jsonl"))
    RESUME_FROM = Path(found[-1]).parent if found else None
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/structured-pruning/study_runs")  # persists across sessions
    CODE_DIR = Path("/content/structured-pruning")
    RESUME_FROM = None  # not needed: WORK_DIR itself lives on Drive
else:
    WORK_DIR = Path("study_runs").resolve()
    CODE_DIR = Path("structured-pruning").resolve()
    RESUME_FROM = None

WORK_DIR.mkdir(parents=True, exist_ok=True)
print("platform:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local")
print("work dir:", WORK_DIR)
print("resume from:", RESUME_FROM)

## 3. Get the code and install only what is missing

In [ ]:
import importlib, subprocess

if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=False)
print(subprocess.run(["git", "-C", str(CODE_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Kaggle and Colab ship transformers/accelerate/sklearn/scipy/matplotlib (and
# usually datasets). Installing only what is missing avoids pulling in a newer
# huggingface-hub that the preinstalled transformers rejects.
needed = {"transformers": "transformers>=4.41", "datasets": "datasets>=2.19",
          "accelerate": "accelerate>=0.30", "sklearn": "scikit-learn", "scipy": "scipy",
          "matplotlib": "matplotlib"}
missing = [spec for mod, spec in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- enable a GPU accelerator!")

## 4. Mechanics check (≈1 minute, no downloads)
Fails fast if the environment is broken, before any GPU time is spent.

In [ ]:
STUDY = CODE_DIR / "experiments" / "research" / "run_study.py"
r = subprocess.run([sys.executable, str(STUDY), "--smoke", "--variants", "quick",
                    "--work-dir", "/tmp/smoke_check"], capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0 and "done:" in r.stdout, r.stderr[-3000:]
print("mechanics OK")

## 5. Run the study
Safe to re-run: finished runs are skipped. Output streams below (one line per run).

In [ ]:
cmd = [sys.executable, "-u", str(STUDY),
       "--tasks", *TASKS, "--model", MODEL,
       "--seeds", *map(str, SEEDS), "--variants", *VARIANTS,
       "--percents", *map(str, PERCENTS),
       "--epochs", str(EPOCHS), "--heal-epochs", str(HEAL_EPOCHS),
       "--work-dir", str(WORK_DIR), "--time-budget-hours", str(TIME_BUDGET_HOURS), *EXTRA_ARGS]
if HEAL_EXAMPLES:
    cmd += ["--heal-examples", str(HEAL_EXAMPLES)]
if RESUME_FROM:
    cmd += ["--resume-from", str(RESUME_FROM)]
print(" ".join(cmd), "\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    # Progress bars arrive as one line per refresh -- thousands on SST-2/QNLI.
    # Keep only their final (100%) line, and drop library warnings.
    if ("%|" in line and "100%" not in line) or "Warning" in line or "warn(" in line:
        continue
    print(line, end="")
proc.wait()
print("exit code", proc.returncode)

## 6. Summarize

In [ ]:
from IPython.display import Image, Markdown, display

SUMMARIZE = CODE_DIR / "experiments" / "research" / "summarize.py"
r = subprocess.run([sys.executable, str(SUMMARIZE), str(WORK_DIR / "results.jsonl"), "--plot"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if (WORK_DIR / "summary.md").exists():
    display(Markdown((WORK_DIR / "summary.md").read_text()))
for png in sorted((WORK_DIR / "plots").glob("*.png")):
    display(Image(filename=str(png)))

## 7. Keep the results

- **Kaggle:** everything under `/kaggle/working/study_runs` becomes this version's **Output**. To continue in a new session, open the notebook editor → **Add Input** → *Your Work* → this notebook (its output), then run all again. The resume source is detected automatically.
- **Colab:** results are already in Google Drive under `MyDrive/structured-pruning/study_runs`.
- Download `results.jsonl`, `environment.jsonl`, `summary.csv`, `summary.md` and `plots/` for the paper. The cached `baselines/` directory is large; it's only needed to continue the study.